In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')   # Modeling/ → pipeline 패키지 import

import numpy as np
import pandas as pd

import config
from pipeline import data_prep, selection, plots

In [ ]:
dd = data_prep.load_disease_filtered()
print(f'Final: {dd.Z_dis.shape[0]} samples, {len(np.unique(dd.dis_pheno))} phenotypes')
print(pd.Series(dd.dis_pheno).value_counts().to_string())

In [ ]:
Z_hc, hc_names = data_prep.load_z_hc()
print(f'Z_hc: {Z_hc.shape}')

In [ ]:
plots.plot_zscore_outlier_hist(dd.Z_dis, dd.dis_pheno)

In [ ]:
all_results, SELECTORS = selection.run_selection(dd.Z_dis, Z_hc, dd.dis_pheno, dd.gene_names)

In [ ]:
global_summary = pd.DataFrame([
    dict(method      = k,
         n_genes     = v['n_genes'],
         macro_sil   = v['macro_sil'],
         macro_mc_auc= v['macro_mc'],
         mean_lr_auc = v['per_pheno']['auc_logreg'].mean(),
         mean_rf_auc = v['per_pheno']['auc_rf'].mean(),
         elapsed_s   = int(v['t']))
    for k, v in all_results.items()
]).set_index('method').sort_values('macro_mc_auc', ascending=False).round(3)

print('══ Global Summary ══')
display(global_summary)

best = global_summary.index[0]
display(all_results[best]['per_pheno'].round(3))

══ Global Summary ══


,n_genes,macro_sil,macro_mc_auc,mean_lr_auc,mean_rf_auc,elapsed_s
method,,,,,,
svd_top30,474,-0.005,0.969,0.978,0.939,10
proportion_top30,527,-0.011,0.965,0.971,0.939,11
effect_size_top30,504,-0.007,0.961,0.973,0.932,10


,n,silhouette,knn_purity,auc_logreg,auc_rf,auc_multiclass
phenotype,,,,,,
CAD_HF+,112,0.003,0.520,0.999,0.999,1.000
CAD_HF-,100,-0.002,0.489,0.999,0.999,1.000
Colorectal Cancer,37,-0.033,0.328,0.998,0.969,0.971
Esophagus Cancer (Chen),25,-0.011,0.235,0.994,0.969,0.987
HIV,13,-0.100,0.072,0.886,0.791,0.921
HIV + Tuberculosis,11,-0.019,0.091,0.974,0.886,0.928
ICI-m,11,-0.062,0.327,1.000,1.000,0.998
ICI-treated Cancer,11,0.231,0.527,1.000,1.000,0.999
Liver Cancer (Chen),10,-0.094,0.080,0.996,0.998,0.947


In [ ]:
# method별 ROC(binary/multiclass) + clustermap/UMAP → CV_Results/Figures/
for key in SELECTORS:
    plots.plot_roc_curves(key, all_results, dd.dis_pheno)
    plots.plot_selection_overview(key, all_results, dd.Z_dis, dd.dis_pheno, dd.dis_names, dd.gene_names)